## **Exercice 1**

In [10]:
logs = [
    " INFO | Auth | User connected ",
    "warning|API|Slow response",
    "ERROR|DB|Connection lost",
    "INFO| |Missing service",
    "DEBUG|Cache| Hit ",
    "invalid line",
    "INFO|Scheduler| Job started ",
    " |API|Missing level",
    "INFO|Worker|",
]
def clean_logs(logs: list[str]) -> list[dict[str, str]]:
    result = []
    for log in logs:
        log = log.strip()
        
        info_log = log.split("|")
        if len(info_log) != 3:
            continue
        
        level, service, message = info_log
        
        if any(info.strip() == "" for info in info_log):
            continue
        item = {
            "level": level.strip().upper(),
            "service": service.strip().lower(),
            "message": message.strip()
        }

        result.append(item)
    return result
clean_logs(logs)

[{'level': 'INFO', 'service': 'auth', 'message': 'User connected'},
 {'level': 'WARNING', 'service': 'api', 'message': 'Slow response'},
 {'level': 'ERROR', 'service': 'db', 'message': 'Connection lost'},
 {'level': 'DEBUG', 'service': 'cache', 'message': 'Hit'},
 {'level': 'INFO', 'service': 'scheduler', 'message': 'Job started'}]

## **Exercice 2**

In [45]:
measurements = [
    {
        "sensor_id": " TEMP-01 ",
        "temperature": 23.5,
        "humidity": 48,
        "status": " OK ",
    },
    {
        "sensor_id": "TEMP-02",
        "temperature": 105,
        "humidity": 40,
        "status": "ok",
    },
    {
        "sensor_id": "",
        "temperature": 18,
        "humidity": 55,
        "status": "warning",
    },
    {
        "sensor_id": "HUM-01",
        "temperature": -12,
        "humidity": 101,
        "status": "OK",
    },
    {
        "sensor_id": " TEMP-03 ",
        "temperature": 31,
        "humidity": 72.5,
        "status": " Warning ",
    },
    {
        "sensor_id": "TEMP-04",
        "temperature": "26",
        "humidity": 50,
        "status": "ok",
    },
    {
        "sensor_id": "TEMP-05",
        "temperature": 19,
        "humidity": 43,
    },
    {
        "sensor_id": "TEMP-06",
        "temperature": 0,
        "humidity": 0,
        "status": "maintenance",
    },
]

def clean_sensor_data(measurements: list[dict]) -> dict[str, list]:
    required_keys = {"sensor_id", "temperature", "humidity", "status"}
    valid_measurements = []
    rejected_positions = []

    def is_number(value: object) -> bool:
        return isinstance(value, (int, float)) and not isinstance(value, bool)

    for position, measurement in enumerate(measurements, start=1):
        if not required_keys.issubset(measurement):
            rejected_positions.append(position)
            continue

        sensor_id = measurement["sensor_id"]
        temperature = measurement["temperature"]
        humidity = measurement["humidity"]
        status = measurement["status"]

        if not isinstance(sensor_id, str) or not isinstance(status, str):
            rejected_positions.append(position)
            continue

        sensor_id = sensor_id.strip()
        status = status.strip().lower()

        is_valid = (
            sensor_id != ""
            and is_number(temperature)
            and -50 <= temperature <= 100
            and is_number(humidity)
            and 0 <= humidity <= 100
            and status in {"ok", "warning"}
        )

        if not is_valid:
            rejected_positions.append(position)
            continue

        valid_measurements.append(
            {
                "sensor_id": sensor_id,
                "temperature": temperature,
                "humidity": humidity,
                "status": status,
                "position": position,
            }
        )

    return {
        "valid": valid_measurements,
        "rejected_positions": rejected_positions,
    }

clean_sensor_data(measurements)

{'valid': [{'sensor_id': 'TEMP-01',
   'temperature': 23.5,
   'humidity': 48,
   'status': 'ok',
   'position': 1},
  {'sensor_id': 'TEMP-03',
   'temperature': 31,
   'humidity': 72.5,
   'status': 'warning',
   'position': 5}],
 'rejected_positions': [2, 3, 4, 6, 7, 8]}

## **Exercice 3**

In [19]:
def merge_tags(
    articles: list[dict[str, str | list[str]]],
) -> dict[str, list[str]]:
    result = {}
    required_keys = {"author", "tags"}

    for article in articles:
        if not required_keys.issubset(article):
            continue

        author_name = article["author"].strip()
        if not author_name:
            continue

        if author_name not in result:
            result[author_name] = []

        for tag in article["tags"]:
            tag_name = tag.strip().lower()

            if not tag_name or tag_name in result[author_name]:
                continue

            result[author_name].append(tag_name)

    return result
# tests
articles = [
    {
        "author": " Alice ",
        "tags": ["Python", " AI ", "python", "", "LLM"],
    },
    {
        "author": "Bob",
        "tags": ["Data", "Python", "data"],
    },
    {
        "author": "Alice",
        "tags": ["FastAPI", "llm", "Python"],
    },
    {
        "author": " ",
        "tags": ["ignored"],
    },
    {
        "author": "Bob",
        "tags": ["SQL", " Python ", ""],
    },
]

merge_tags(articles)

{'Alice': ['python', 'ai', 'llm', 'fastapi'], 'Bob': ['data', 'python', 'sql']}

## **Exercice 4**

In [30]:
def clean_commands(
    commands: list[dict[str, float|int|str]]
) -> list[dict[str, float|int|str]]:
    result = []
    required_keys = {"customer_id", "amount", "status"}
    for command in commands:
        if not required_keys.issubset(command):
            continue
        
        if not isinstance(command["amount"], (int, float)):
            continue
        
        if command["status"].strip() != "paid":
            continue
        
        result.append(command)
    return result

def aggregate_commands(
    commands: list[dict[str, float|int|str]]
) -> dict[int, dict[str, float|int]]:
    # clean commands:
    cleaned_commands = clean_commands(commands)

    # Aggregation
    result = {}
    for command in cleaned_commands:
        customer_id = command["customer_id"]
        amount = command["amount"]

        if customer_id not in result:
            result[customer_id] = {}
        
        result[customer_id] = {
            "count": result[customer_id].get("count", 0) + 1,
            "total": result[customer_id].get("total", 0) + amount,
            "average": (result[customer_id].get("total", 0) + amount) / (result[customer_id].get("count", 0) + 1)                                                   
        }
    return result

# Test
orders = [
    {"customer_id": 101, "amount": 25.5, "status": "paid"},
    {"customer_id": 102, "amount": 10, "status": "pending"},
    {"customer_id": 101, "amount": 50, "status": "paid"},
    {"customer_id": 103, "amount": "18", "status": "paid"},
    {"customer_id": 102, "amount": 40, "status": "paid"},
    {"customer_id": 101, "amount": 15, "status": "cancelled"},
    {"customer_id": 102, "amount": 60, "status": "paid"},
    {"customer_id": 104, "status": "paid"},
]

aggregate_commands(orders)

{101: {'count': 2, 'total': 75.5, 'average': 37.75},
 102: {'count': 2, 'total': 100, 'average': 50.0}}

## **Exercice 5**

In [35]:
def sort_matrix(matrix: list[list[int]]) -> list[list[int]]:
    sorted_matrix = []
    for row in matrix:
        if not row:
            continue
        sorted_matrix.append(sorted(row,reverse=False))
    return sorted_matrix
# Test
matrix = [
    [5, 2, 8, 1],
    [],
    [7, 7, 3],
    [10],
    [9, 4, 6],
]
sort_matrix(matrix)

[[1, 2, 5, 8], [3, 7, 7], [10], [4, 6, 9]]

## **Exercice 6**

In [44]:
def merge_measure(sensor_a: list[int], sensor_b: list[int]) -> list[int]:
    if len(sensor_a) != len(sensor_b):
        return []
    else:
        result = []
        for (a,b) in zip(sensor_a, sensor_b):
            result.append((a + b)/2)
    return result
# test
sensor_a = [20, 22, 18, 25, 24]
sensor_b = [18, 24, 20, 21, 26]
assert merge_measure(sensor_a, sensor_b) == [19.0, 23.0, 19.0, 23.0, 25.0]
sensor_a = [1, 2, 3]
sensor_b = [4, 5]
assert merge_measure(sensor_a, sensor_b) == []

## **Exercice 7**

In [14]:
def agg_measurements(
    measurements: list[tuple[int, list[int]]]
) -> dict[int, dict[str, int | float | list[int]]]:
    agg_datas = {}

    for machine_id, measures in measurements:
        if not measures:
            continue

        total = sum(measures)
        count = len(measures)
        average = total / count

        agg_datas[machine_id] = {
            "count": count,
            "min": min(measures),
            "max": max(measures),
            "total": total,
            "average": average,
            "above_average": sum(val > average for val in measures),
            "sorted_values": sorted(measures),
        }

    return agg_datas

# test
measurements = [
    (101, [12, 8, 15, 10]),
    (102, [5, 5, 5]),
    (103, []),
    (104, [20, -5, 10, 25, 0]),
    (105, [7]),
]

agg_datas = agg_measurements(measurements)
assert agg_datas[101] == {
    "count": 4,
    "min": 8,
    "max": 15,
    "total": 45,
    "average": 11.25,
    "above_average": 2,
    "sorted_values": [8, 10, 12, 15],
}
assert 103 not in agg_datas
assert agg_datas[102]['min'] == agg_datas[102]['max']
assert agg_datas[105]['above_average'] == 0

## **Exercice 8**

In [34]:
class Inventory:
    def __init__(self):
        self.stock = {}
    def add(self, product_id:int, quantity:int) -> bool:
        if quantity <= 0:
            return False
        
        if product_id not in self.stock:
            self.stock[product_id] = quantity
        else:
            self.stock[product_id] += quantity
        return True
    
    def withdraw(self, product_id:int, quantity:int) -> bool:
        if quantity <= 0:
            return False
        if product_id not in self.stock:
            return False
        elif self.stock[product_id] < quantity:
            return False
        else:
            self.stock[product_id] -= quantity
            return True

    def quantity(self, product_id:int) -> int:
        if product_id not in self.stock:
            return 0
        return self.stock[product_id]

    def total_quantity(self) -> int:
        return sum(self.stock.values())

    def out_of_stock_products(self) -> list[int]:
        products = []
        for product_id in self.stock:
            if self.quantity(product_id) == 0:
                products.append(product_id)
        return sorted(products, reverse=False)
# Tests
inventory = Inventory()

inventory.add(101, 5)
inventory.add(102, 3)
inventory.withdraw(101, 2)

assert inventory.quantity(101) == 3
assert inventory.total_quantity() == 6
assert inventory.quantity(999) == 0
assert inventory.withdraw(999, 2) is False
assert inventory.add(103, 0) is False
assert inventory.withdraw(101, -2) is False

## Exercice 9 — Normalisation d’une matrice de mesures

In [13]:
import numpy as np

def process_measurements(X: np.ndarray) -> dict:
    column_mean = X.mean(axis=0)
    column_min = X.min(axis=0)
    column_max = X.max(axis=0)
    column_range = column_max - column_min
    row_mean = X.mean(axis=1)

    result = {
        "column_mean": column_mean,
        "column_min": column_min,
        "column_max": column_max,
        "column_range": column_range,
        "normalized": (X - column_min) / column_range,
        "selected_rows": X[X[:, 0] > column_mean[0]],
        "row_mean": row_mean,
        "best_row": np.argmax(row_mean),
    }

    return result
                                   
# Tests
measurements = np.array([
    [10.0, 100.0, 5.0],
    [20.0, 120.0, 7.0],
    [30.0, 110.0, 6.0],
    [40.0, 140.0, 8.0],
    [50.0, 130.0, 9.0],
])

# Tests

result = process_measurements(measurements)

assert np.allclose(
    result["column_mean"],
    [30.0, 120.0, 7.0]
)

assert np.allclose(
    result["column_min"],
    [10.0, 100.0, 5.0]
)

assert np.allclose(
    result["column_max"],
    [50.0, 140.0, 9.0]
)

assert np.allclose(
    result["column_range"],
    [40.0, 40.0, 4.0]
)

assert np.allclose(
    result["normalized"],
    [
        [0.00, 0.00, 0.00],
        [0.25, 0.50, 0.50],
        [0.50, 0.25, 0.25],
        [0.75, 1.00, 0.75],
        [1.00, 0.75, 1.00],
    ]
)

assert np.array_equal(
    result["selected_rows"],
    [
        [40.0, 140.0, 8.0],
        [50.0, 130.0, 9.0],
    ]
)

assert np.allclose(
    result["row_mean"],
    [
        115 / 3,
        147 / 3,
        146 / 3,
        188 / 3,
        189 / 3,
    ]
)

assert result["best_row"] == 4

In [7]:
import numpy as np
X = np.array([
    [10.0, 100.0, 5.0],
    [20.0, 120.0, 7.0],
    [30.0, 110.0, 6.0],
    [40.0, 140.0, 8.0],
    [50.0, 130.0, 9.0],
])
X[X[:,0] > X[:,0].mean()]

array([[ 40., 140.,   8.],
       [ 50., 130.,   9.]])

## Transformation d’un batch de scores

In [26]:
def process_scores(scores: np.ndarray) -> dict:
    clipped =  np.clip(scores, 10, 30)
    column_median = np.median(scores, axis=0)
    column_std = scores.std(axis=0)
    row_sum = scores.sum(axis=1)
    high_values = scores >= 25
    replaced = np.where(scores < 10, 0, scores)
    transposed = np.transpose(scores)
    flat = scores.flatten()
    best_columns = np.argmax(scores, axis=1)

    return {
        "clipped": clipped,
        "column_median": column_median,
        "column_std": column_std,
        "row_sum": row_sum,
        "high_values": high_values,
        "replaced": replaced,
        "transposed": transposed,
        "flat": flat,
        "best_columns": best_columns
    }

# Tests
scores = np.array([
    [12, 18, 25, 7],
    [30, 22, 15, 19],
    [8, 10, 14, 6],
    [21, 27, 24, 18],
    [35, 32, 29, 31],
], dtype=float)
result = process_scores(scores)

assert np.array_equal(
    result["clipped"],
    np.array([
        [12, 18, 25, 10],
        [30, 22, 15, 19],
        [10, 10, 14, 10],
        [21, 27, 24, 18],
        [30, 30, 29, 30],
    ], dtype=float)
)

assert np.allclose(
    result["column_median"],
    [21, 22, 24, 18]
)

assert np.allclose(
    result["row_sum"],
    [62, 86, 38, 90, 127]
)

assert np.array_equal(
    result["high_values"],
    np.array([
        [False, False, True,  False],
        [True,  False, False, False],
        [False, False, False, False],
        [False, True,  False, False],
        [True,  True,  True,  True],
    ])
)

assert np.array_equal(
    result["replaced"],
    np.array([
        [12, 18, 25, 0],
        [30, 22, 15, 19],
        [0, 10, 14, 0],
        [21, 27, 24, 18],
        [35, 32, 29, 31],
    ], dtype=float)
)

assert result["transposed"].shape == (4, 5)

assert result["flat"].shape == (20,)

assert np.array_equal(
    result["best_columns"],
    [2, 0, 2, 1, 0]
)

# Vérifie que l'entrée n'a pas été modifiée
assert np.array_equal(
    scores,
    np.array([
        [12, 18, 25, 7],
        [30, 22, 15, 19],
        [8, 10, 14, 6],
        [21, 27, 24, 18],
        [35, 32, 29, 31],
    ], dtype=float)
)